download the dataset

In [1]:
!pip install --upgrade gdown
import gdown

folder_url = 'https://drive.google.com/drive/folders/1s5W0Nfz9NEN8gP14tge8GuouUDXs2Ssq'
gdown.download_folder(folder_url, quiet=False, use_cookies=False)

  Attempting uninstall: gdown
    Found existing installation: gdown 5.2.2
    Uninstalling gdown-5.2.2:
      Successfully uninstalled gdown-5.2.2


Retrieving folder contents


Processing file 1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF full-shape.zip
Processing file 1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p partial.zip
Processing file 1vwFQbYRDQlyxOZk0N0Hlxi8O_bhHxgIM rotate.zip


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF
From (redirected): https://drive.google.com/uc?id=1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF&confirm=t&uuid=e114a187-9d2e-4648-be1c-f3903dcf54e9
To: /content/AffordanceNet/full-shape.zip
100%|██████████| 558M/558M [00:08<00:00, 62.8MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p
From (redirected): https://drive.google.com/uc?id=1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p&confirm=t&uuid=faf72cd5-720f-4e88-a596-2d97b1b7e8eb
To: /content/AffordanceNet/partial.zip
100%|██████████| 2.19G/2.19G [00:35<00:00, 61.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1vwFQbYRDQlyxOZk0N0Hlxi8O_bhHxgIM
To: /content/AffordanceNet/rotate.zip
100%|██████████| 1.15M/1.15M [00:00<00:00, 150MB/s]
Download completed


['/content/AffordanceNet/full-shape.zip',
 '/content/AffordanceNet/partial.zip',
 '/content/AffordanceNet/rotate.zip']

In [2]:
# Unzip all files in the AffordanceNet folder
!unzip -o /content/AffordanceNet/full-shape.zip -d /content/AffordanceNet/
!unzip -o /content/AffordanceNet/partial.zip -d /content/AffordanceNet/
!unzip -o /content/AffordanceNet/rotate.zip -d /content/AffordanceNet/

# List the directory to see the extracted content
!ls -R /content/AffordanceNet/

Archive:  /content/AffordanceNet/full-shape.zip
  inflating: /content/AffordanceNet/full_shape_train_data.pkl  
  inflating: /content/AffordanceNet/full_shape_val_data.pkl  
Archive:  /content/AffordanceNet/partial.zip
  inflating: /content/AffordanceNet/partial_train_data.pkl  
  inflating: /content/AffordanceNet/partial_val_data.pkl  
Archive:  /content/AffordanceNet/rotate.zip
  inflating: /content/AffordanceNet/rotate_val_data.pkl  
/content/AffordanceNet/:
full_shape_train_data.pkl  partial_train_data.pkl  rotate_val_data.pkl
full_shape_val_data.pkl    partial_val_data.pkl    rotate.zip
full-shape.zip		   partial.zip


In [3]:
# Remove all zip files in the AffordanceNet directory
!rm /content/AffordanceNet/*.zip

# List the directory to confirm they are gone
!ls /content/AffordanceNet/

full_shape_train_data.pkl  partial_train_data.pkl  rotate_val_data.pkl
full_shape_val_data.pkl    partial_val_data.pkl


In [1]:
!pip install torch torchvision
!pip install numpy scipy open3d scikit-learn matplotlib pandas tqdm
!pip install tensorboardX

  Using cached tensorboardx-2.6.5-py3-none-any.whl.metadata (6.6 kB)
Using cached tensorboardx-2.6.5-py3-none-any.whl (87 kB)


In [2]:
import pickle

#from google.colab import drive
#drive.mount('/content/drive')

# Update these paths to match your Google Drive
TRAIN_PKL = "/content/AffordanceNet/full_shape_train_data.pkl"
VAL_PKL   = "/content/AffordanceNet/full_shape_val_data.pkl"

# Load data
with open(TRAIN_PKL, "rb") as f:
    train_data = pickle.load(f)

with open(VAL_PKL, "rb") as f:
    val_data = pickle.load(f)

print(f"Train samples: {len(train_data)}")
print(f"Val samples: {len(val_data)}")

# Extract affordance names from first sample
sample = train_data[0]
AFFORDANCE_NAMES = sample['affordance']
print(f"All affordance classes: {AFFORDANCE_NAMES}")

Train samples: 16082
Val samples: 2285
All affordance classes: ['grasp', 'contain', 'lift', 'openable', 'layable', 'sittable', 'support', 'wrap_grasp', 'pourable', 'move', 'displaY', 'pushable', 'pull', 'listen', 'wear', 'press', 'cut', 'stab']


In [3]:
from collections import defaultdict


# Count positive points per class across all training samples
class_counts = defaultdict(int)
total_points = 0

for sample in train_data:
    label_dict = sample['full_shape']['label']
    for aff in AFFORDANCE_NAMES:
        mask = label_dict[aff].flatten()
        pos = (mask > 0.5).sum()
        class_counts[aff] += pos
    total_points += len(mask)

# Sort and pick top 5
sorted_classes = sorted(AFFORDANCE_NAMES, key=lambda x: class_counts[x], reverse=True)
SELECTED_CLASSES = sorted_classes[:5]
SELECTED_INDICES = [AFFORDANCE_NAMES.index(c) for c in SELECTED_CLASSES]
NUM_CLASSES = len(SELECTED_CLASSES)

print(f"Selected classes: {SELECTED_CLASSES}")
print(f"Indices: {SELECTED_INDICES}")
print(f"Number of classes: {NUM_CLASSES}")

# Create zero-shot split by semantic class
all_categories = set(s['semantic class'] for s in train_data)
TEST_CATEGORIES = ['Scissors', 'Hammer', 'Laptop', 'Camera']
TEST_CATEGORIES = [c for c in TEST_CATEGORIES if c in all_categories]

train_split = [s for s in train_data if s['semantic class'] not in TEST_CATEGORIES]
test_split = [s for s in train_data if s['semantic class'] in TEST_CATEGORIES]

print(f"Train categories: {len(set(s['semantic class'] for s in train_split))}")
print(f"Test categories (zero-shot): {len(set(s['semantic class'] for s in test_split))}")

Selected classes: ['move', 'support', 'sittable', 'contain', 'displaY']
Indices: [9, 6, 5, 1, 10]
Number of classes: 5
Train categories: 21
Test categories (zero-shot): 2


In [4]:
# Filter training samples to only those with at least one positive point for selected classes
filtered_train = []

for sample in train_split:
    label_dict = sample['full_shape']['label']
    has_positive = False
    for idx_c in SELECTED_INDICES:
        aff = AFFORDANCE_NAMES[idx_c]
        mask = label_dict[aff].flatten()
        if (mask > 0.5).sum() > 0:
            has_positive = True
            break
    if has_positive:
        filtered_train.append(sample)

print(f"Original train samples: {len(train_split)}")
print(f"Filtered train samples: {len(filtered_train)}")
print(f"Test samples (zero-shot): {len(test_split)}")

Original train samples: 15738
Filtered train samples: 14369
Test samples (zero-shot): 344


In [5]:
# ------------------------------------------------------------
# Oversample samples containing rare classes
# ------------------------------------------------------------
rare_class_indices = [2, 3, 4]  # sittable, contain, display (0-indexed)
oversampled_train = []
for sample in filtered_train:
    label_dict = sample['full_shape']['label']
    has_rare = False
    for idx in rare_class_indices:
        aff = AFFORDANCE_NAMES[idx]
        if (label_dict[aff].flatten() > 0.5).any():
            has_rare = True
            break
    oversampled_train.append(sample)
    if has_rare:
        # duplicate 5 times (you can increase to 8-10 later if needed)
        for _ in range(5):
            oversampled_train.append(sample)

print(f"Original train samples: {len(filtered_train)}")
print(f"Oversampled train samples: {len(oversampled_train)}")

Original train samples: 14369
Oversampled train samples: 24779


In [8]:
# ------------------------------------------------------------
# Create datasets (using oversampled_train for training)
# ------------------------------------------------------------
train_dataset = AffordanceMultiLabelDataset(
    oversampled_train,          # the list after oversampling
    selected_indices=SELECTED_INDICES,
    num_points=2048,
    occlusion=False,            # keep occlusion off for now
    crop_angle=120,
    dropout=0.0
)

val_dataset = AffordanceMultiLabelDataset(
    val_data,
    selected_indices=SELECTED_INDICES,
    num_points=2048,
    occlusion=False,
    crop_angle=0,
    dropout=0.0
)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Val dataset size: {len(val_dataset)}")

Train dataset size: 24779
Val dataset size: 2285


In [7]:
import torch
from torch.utils.data import Dataset
import numpy as np

class AffordanceMultiLabelDataset(Dataset):
    def __init__(self, data_list, selected_indices, num_points=2048,
                 occlusion=True, crop_angle=120, dropout=0.3):
        self.data = data_list
        self.selected_indices = selected_indices
        self.num_points = num_points
        self.occlusion = occlusion
        self.crop_angle = crop_angle
        self.dropout = dropout

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        coords = sample['full_shape']['coordinate']
        label_dict = sample['full_shape']['label']

        # Extract selected classes and BINARIZE the masks
        masks = []
        for idx_c in self.selected_indices:
            aff = AFFORDANCE_NAMES[idx_c]
            mask = label_dict[aff].flatten()
            # Hard threshold to 0 or 1 (critical fix)
            mask = (mask > 0.5).astype(np.float32)
            masks.append(mask)
        labels = np.stack(masks, axis=1)

        # Apply occlusion (preserves positives)
        if self.occlusion:
            coords, labels = self.simulate_occlusion(coords, labels)

        # If after occlusion there are zero points, fallback
        if len(coords) == 0:
            coords = sample['full_shape']['coordinate']
            masks = []
            for idx_c in self.selected_indices:
                aff = AFFORDANCE_NAMES[idx_c]
                mask = label_dict[aff].flatten()
                mask = (mask > 0.5).astype(np.float32)
                masks.append(mask)
            labels = np.stack(masks, axis=1)

        # Subsample while preserving positives
        if len(coords) > self.num_points:
            pos_mask = (labels.sum(axis=1) > 0.5)
            pos_indices = np.where(pos_mask)[0]
            neg_indices = np.where(~pos_mask)[0]
            n_pos = len(pos_indices)
            n_neg_needed = max(0, self.num_points - n_pos)

            if n_neg_needed > 0 and len(neg_indices) > 0:
                neg_selected = np.random.choice(neg_indices, n_neg_needed, replace=False)
                selected_indices = np.concatenate([pos_indices, neg_selected])
            elif n_pos > 0:
                selected_indices = pos_indices
            else:
                selected_indices = np.random.choice(len(coords), self.num_points, replace=False)

            if len(selected_indices) < self.num_points:
                remaining = self.num_points - len(selected_indices)
                extra = np.random.choice(len(coords), remaining, replace=True)
                selected_indices = np.concatenate([selected_indices, extra])
        else:
            selected_indices = np.random.choice(len(coords), self.num_points, replace=True)

        coords = coords[selected_indices]
        labels = labels[selected_indices]

        return torch.tensor(coords, dtype=torch.float32), torch.tensor(labels, dtype=torch.float32)

    def simulate_occlusion(self, points, labels):
        """Always keep positive points, apply occlusion only to negatives."""
        if len(points) == 0:
            return points, labels

        # Identify positive points
        pos_mask = (labels.sum(axis=1) > 0.5)
        pos_indices = np.where(pos_mask)[0]
        neg_indices = np.where(~pos_mask)[0]

        # Occlude negative points only
        if len(neg_indices) > 0:
            neg_points = points[neg_indices]
            centroid = np.mean(neg_points, axis=0)
            centered = neg_points - centroid
            theta = np.random.uniform(0, 2*np.pi)
            phi = np.random.uniform(0, np.pi)
            view_dir = np.array([np.sin(phi)*np.cos(theta), np.sin(phi)*np.sin(theta), np.cos(phi)])
            norms = np.linalg.norm(centered, axis=1)
            norms[norms == 0] = 1e-8
            dirs = centered / norms[:, np.newaxis]
            angles = np.arccos(np.clip(np.dot(dirs, view_dir), -1, 1))
            mask = angles < np.radians(self.crop_angle/2)
            if self.dropout > 0:
                keep_neg = np.where(mask)[0]
                n_keep = int(len(keep_neg) * (1 - self.dropout))
                if n_keep > 0 and len(keep_neg) > 0:
                    keep_neg = np.random.choice(keep_neg, n_keep, replace=False)
                    mask = np.zeros(len(neg_points), dtype=bool)
                    mask[keep_neg] = True
            neg_points = neg_points[mask]
            neg_labels = labels[neg_indices][mask]
        else:
            neg_points = np.empty((0, 3))
            neg_labels = np.empty((0, labels.shape[1]))

        # Always keep positive points
        pos_points = points[pos_indices]
        pos_labels = labels[pos_indices]

        # Combine
        if len(pos_points) > 0:
            coords = np.concatenate([pos_points, neg_points], axis=0)
            labels = np.concatenate([pos_labels, neg_labels], axis=0)
        else:
            coords = neg_points
            labels = neg_labels

        return coords, labels

In [9]:
import torch.nn as nn


def knn(x, k=20):
    B, _, N = x.shape
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x**2, dim=1, keepdim=True)
    pairwise_distance = -xx - inner - xx.transpose(2, 1)
    idx = pairwise_distance.topk(k=k, dim=-1)[1]
    return idx

def get_graph_feature(x, k=20, idx=None):
    B, C, N = x.shape
    if idx is None:
        idx = knn(x, k=k)
    device = x.device
    idx_base = torch.arange(0, B, device=device).view(-1, 1, 1) * N
    idx = idx + idx_base
    idx = idx.view(-1)
    _, num_dims, _ = x.size()
    x = x.transpose(2, 1).contiguous()
    feature = x.view(B * N, -1)[idx, :]
    feature = feature.view(B, N, k, C)
    x = x.view(B, N, 1, C).repeat(1, 1, k, 1)
    feature = torch.cat((feature - x, x), dim=3)
    feature = feature.permute(0, 3, 1, 2).contiguous()
    return feature

class DGCNNPartSeg(nn.Module):
    def __init__(self, num_classes=5, k=20):
        super().__init__()
        self.k = k
        self.conv1 = nn.Sequential(
            nn.Conv2d(6, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(64*2, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64*2, 128, kernel_size=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )
        self.fc = nn.Sequential(
            nn.Conv1d(64 + 64 + 128 + 256, 256, 1, bias=False),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Conv1d(256, 128, 1, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Conv1d(128, num_classes, 1)
        )

    def forward(self, x):
        B, C, N = x.shape
        edge_feat = get_graph_feature(x, k=self.k)
        x1 = self.conv1(edge_feat).max(dim=-1, keepdim=False)[0]
        edge_feat = get_graph_feature(x1, k=self.k)
        x2 = self.conv2(edge_feat).max(dim=-1, keepdim=False)[0]
        edge_feat = get_graph_feature(x2, k=self.k)
        x3 = self.conv3(edge_feat).max(dim=-1, keepdim=False)[0]
        x_cat = torch.cat([x1, x2, x3], dim=1)
        x_global = x_cat.max(dim=-1, keepdim=True)[0].repeat(1, 1, N)
        x_concat = torch.cat([x1, x2, x3, x_global], dim=1)
        out = self.fc(x_concat).permute(0, 2, 1)
        return out

add on

In [10]:
import torch

# Compute per-class positive ratio
class_pos_ratios = {}
for aff in SELECTED_CLASSES:
    ratio = class_counts[aff] / total_points
    class_pos_ratios[aff] = ratio

# Stronger weights: exponent 0.7, cap at 20
max_ratio = max(class_pos_ratios.values())
weights = []
for aff in SELECTED_CLASSES:
    w = (max_ratio / (class_pos_ratios[aff] + 1e-8)) ** 0.7   # higher exponent
    weights.append(min(w, 20.0))   # cap raised to 20

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights_tensor = torch.tensor(weights, dtype=torch.float32).to(device)
print("Per-class weights:", weights)

Per-class weights: [np.float64(0.9999998588545391), np.float64(1.0712978887178186), np.float64(1.4427416297164943), np.float64(2.2899782164437923), np.float64(3.28127027715093)]


In [11]:
import numpy as np
from collections import defaultdict
from torch.utils.data import Sampler, DataLoader

# 1. Pre-compute which classes each sample contains
sample_classes = []   # list of lists: for each sample, the classes present
for i in range(len(train_dataset)):
    _, labels = train_dataset[i]          # labels shape (N, C)
    present = (labels.sum(dim=0) > 0.5).nonzero(as_tuple=False).flatten().tolist()
    sample_classes.append(present)

class_to_indices = defaultdict(list)
for idx, classes_present in enumerate(sample_classes):
    for c in classes_present:
        class_to_indices[c].append(idx)

# 2. Verify all classes have enough samples
for c in range(NUM_CLASSES):
    print(f"Class {c} ({SELECTED_CLASSES[c]}): {len(class_to_indices[c])} samples")

# 3. Custom sampler – every batch includes at least one sample from each class
class BalancedBatchSampler(Sampler):
    def __init__(self, sample_classes, batch_size, num_classes, class_to_indices):
        self.sample_classes = sample_classes
        self.batch_size = batch_size
        self.num_classes = num_classes
        self.class_to_indices = class_to_indices
        self.sample_to_class_set = [set(sc) for sc in sample_classes]
        self.indices = list(range(len(sample_classes)))
        self.num_batches = len(self.indices) // batch_size

    def __iter__(self):
        shuffled = np.random.permutation(self.indices).tolist()
        batches = []
        for _ in range(self.num_batches):
            batch = []
            # Guarantee one sample per class
            for c in range(self.num_classes):
                for idx in shuffled:
                    if c in self.sample_to_class_set[idx]:
                        batch.append(idx)
                        shuffled.remove(idx)
                        break
            # Fill the rest of the batch
            needed = self.batch_size - len(batch)
            if needed > 0:
                extra = shuffled[:needed]
                batch.extend(extra)
                shuffled = shuffled[needed:]
            batches.append(batch)
        np.random.shuffle(batches)
        for batch in batches:
            yield batch

    def __len__(self):
        return self.num_batches

# 4. Use the balanced sampler
batch_size = 8
train_sampler = BalancedBatchSampler(sample_classes, batch_size, NUM_CLASSES, class_to_indices)
train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=2)

# Keep val_loader as is (using standard shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

Class 0 (move): 9732 samples
Class 1 (support): 10429 samples
Class 2 (sittable): 4956 samples
Class 3 (contain): 13163 samples
Class 4 (displaY): 990 samples


In [12]:
import numpy as np
from collections import defaultdict
from torch.utils.data import Sampler, DataLoader

# 1. Pre-compute which classes each training sample contains
sample_classes = []   # list of lists: for each sample, the class indices present
for i in range(len(train_dataset)):
    _, labels = train_dataset[i]          # shape (N, C)
    present = (labels.sum(dim=0) > 0.5).nonzero(as_tuple=False).flatten().tolist()
    sample_classes.append(present)

class_to_indices = defaultdict(list)
for idx, classes_present in enumerate(sample_classes):
    for c in classes_present:
        class_to_indices[c].append(idx)

# 2. Print how many samples contain each class (sanity check)
for c in range(NUM_CLASSES):
    print(f"Class {c} ({SELECTED_CLASSES[c]}): {len(class_to_indices[c])} samples")

# 3. Balanced batch sampler – guarantees one sample per class per batch
class BalancedBatchSampler(Sampler):
    def __init__(self, sample_classes, batch_size, num_classes, class_to_indices):
        self.sample_classes = sample_classes
        self.batch_size = batch_size
        self.num_classes = num_classes
        self.class_to_indices = class_to_indices
        self.sample_to_class_set = [set(sc) for sc in sample_classes]
        self.indices = list(range(len(sample_classes)))
        self.num_batches = len(self.indices) // batch_size

    def __iter__(self):
        shuffled = np.random.permutation(self.indices).tolist()
        batches = []
        for _ in range(self.num_batches):
            batch = []
            # 1) Pick one sample for each class
            for c in range(self.num_classes):
                # Find a sample that contains class c and is still available
                for idx in shuffled:
                    if c in self.sample_to_class_set[idx]:
                        batch.append(idx)
                        shuffled.remove(idx)
                        break
                # If no such sample exists (shouldn't happen if class has samples), skip
            # 2) Fill the rest of the batch with random remaining samples
            needed = self.batch_size - len(batch)
            if needed > 0:
                extra = shuffled[:needed]
                batch.extend(extra)
                shuffled = shuffled[needed:]
            batches.append(batch)
        np.random.shuffle(batches)
        for batch in batches:
            yield batch

    def __len__(self):
        return self.num_batches

# 4. Create the training loader using the sampler
batch_size = 8
train_sampler = BalancedBatchSampler(sample_classes, batch_size, NUM_CLASSES, class_to_indices)
train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=2)

# Validation loader remains unchanged (no sampler, just shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

Class 0 (move): 9732 samples
Class 1 (support): 10427 samples
Class 2 (sittable): 4955 samples
Class 3 (contain): 13164 samples
Class 4 (displaY): 990 samples


In [13]:
import torch.optim as optim
from tqdm import tqdm
import torch.nn as nn
import torch
import numpy as np

# ------------------------------------------------------------
# Simple BCE + Dice loss (multi‑label independent)
# ------------------------------------------------------------
class SimpleMultiLabelLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        # pred, target: (B, N, C)
        bce = nn.functional.binary_cross_entropy_with_logits(pred, target, reduction='none')
        bce = bce.mean()   # mean over all points and classes

        # Dice per class, then mean
        pred_sig = torch.sigmoid(pred)
        dice_loss = 0.0
        for c in range(pred.size(2)):
            inter = (pred_sig[..., c] * target[..., c]).sum()
            union = pred_sig[..., c].sum() + target[..., c].sum()
            dice = (2. * inter + self.smooth) / (union + self.smooth)
            dice_loss += (1 - dice)
        dice_loss /= pred.size(2)

        return bce + dice_loss

# ------------------------------------------------------------
# Setup
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DGCNNPartSeg(num_classes=NUM_CLASSES, k=20).to(device)

# Mild positive bias (0.5 is enough, 1.0 also fine)
with torch.no_grad():
    model.fc[-1].bias.data += 0.5

criterion = SimpleMultiLabelLoss(smooth=1.0)
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

NUM_EPOCHS = 50
best_val_iou = 0.0

# ------------------------------------------------------------
# Training loop
# ------------------------------------------------------------
for epoch in range(NUM_EPOCHS):
    # --- Train ---
    model.train()
    total_loss = 0
    for points, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Train"):
        points = points.permute(0, 2, 1).to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        pred = model(points)
        loss = criterion(pred, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} Train Loss: {avg_loss:.4f}")

    # --- Validation (threshold=0.5) ---
    model.eval()
    iou_per_class = torch.zeros(NUM_CLASSES).to(device)
    count_per_class = torch.zeros(NUM_CLASSES).to(device)
    val_loss = 0.0

    with torch.no_grad():
        for points, labels in tqdm(val_loader, desc="Validating"):
            points = points.permute(0, 2, 1).to(device)
            labels = labels.to(device)
            pred = model(points)
            loss = criterion(pred, labels)
            val_loss += loss.item()

            pred_binary = (torch.sigmoid(pred) > 0.5).long()
            labels_binary = labels.long()

            for b in range(points.shape[0]):
                for c in range(NUM_CLASSES):
                    if labels_binary[b, :, c].sum() > 0:
                        inter = (pred_binary[b, :, c] & labels_binary[b, :, c]).sum().float()
                        union = (pred_binary[b, :, c] | labels_binary[b, :, c]).sum().float()
                        iou = inter / (union + 1e-8)
                        iou_per_class[c] += iou
                        count_per_class[c] += 1

    avg_iou_per_class = iou_per_class / (count_per_class + 1e-8)
    mean_iou = avg_iou_per_class.mean().item()
    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch {epoch+1} Val Loss: {avg_val_loss:.4f}, mIoU: {mean_iou:.6f}")
    print("Per-class IoU:", [f"{x:.4f}" for x in avg_iou_per_class.tolist()])

    scheduler.step()

    if mean_iou > best_val_iou:
        best_val_iou = mean_iou
        torch.save(model.state_dict(), "/content/output/best_model_weighted.pth")
        print(f"✅ New best saved! mIoU: {best_val_iou:.4f}")

print(f"\n🏆 Best mIoU: {best_val_iou:.4f}")

Epoch 1 Train: 100%|██████████| 3097/3097 [07:05<00:00,  7.28it/s]


Epoch 1 Train Loss: 0.6847


Validating: 100%|██████████| 286/286 [00:26<00:00, 10.82it/s]


Epoch 1 Val Loss: 0.6506, mIoU: 0.313430
Per-class IoU: ['0.1656', '0.2793', '0.5700', '0.3168', '0.2356']
✅ New best saved! mIoU: 0.3134


Epoch 2 Train: 100%|██████████| 3097/3097 [07:07<00:00,  7.25it/s]


...

Epoch 21 Train Loss: 0.5069


Validating: 100%|██████████| 286/286 [00:26<00:00, 10.91it/s]


Epoch 21 Val Loss: 0.5099, mIoU: 0.357501
Per-class IoU: ['0.2099', '0.3218', '0.6034', '0.3454', '0.3070']


Epoch 22 Train:   1%|▏         | 39/3097 [00:07<10:26,  4.88it/s]


KeyboardInterrupt: 